# Practical 8 — Spam or Ham Detection Using NLP

## Objective
Build a text classification pipeline to classify messages as spam or ham using TF-IDF and Multinomial Naive Bayes.

**How to use:** Put the required CSV file(s) in the `data/` folder, update the filename/path if needed, and run cells from top to bottom. Where a dataset is unavailable, the notebook reports that clearly rather than inventing results.

## Imports

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

## Load a labeled SMS dataset

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

csv_path = DATA_DIR / "spam.csv"
if not csv_path.exists():
    raise FileNotFoundError("Add a labeled spam.csv to data/ with columns label and message (or v1 and v2 from common SMS spam datasets).")
sms = pd.read_csv(csv_path, encoding="latin-1")
if {"v1", "v2"}.issubset(sms.columns):
    sms = sms.rename(columns={"v1":"label", "v2":"message"})
elif not {"label", "message"}.issubset(sms.columns):
    raise KeyError("CSV needs label/message columns, or v1/v2 columns.")
sms = sms[["label", "message"]].dropna()
sms["label"] = sms["label"].astype(str).str.lower().str.strip()
sms = sms[sms["label"].isin(["spam", "ham"])]
display(sms.head())
print(sms["label"].value_counts())

## Train and evaluate

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    sms["message"], sms["label"], test_size=0.2, random_state=42, stratify=sms["label"]
)
spam_model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), min_df=1)),
    ("classifier", MultinomialNB())
])
spam_model.fit(X_train, y_train)
pred = spam_model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=["ham", "spam"], cmap=None)
plt.title("Spam/Ham Confusion Matrix")
plt.tight_layout()
plt.show()

## Try a message

In [ ]:
sample_message = input("Enter a message to classify: ")
print("Prediction:", spam_model.predict([sample_message])[0])

## Conclusion

Report test accuracy and class-level precision/recall. Do not judge a spam model on accuracy alone when spam and ham counts are imbalanced.